# Middletown Township School District — AMR Analysis

Extract and analyze data from the Auditor's Management Reports (AMR) for fiscal years 2019–2025.

In [1]:
from pathlib import Path

import pandas as pd
import pdfplumber

PROJECT_ROOT = Path("/Users/loganbest/Desktop/Projects/Middletown_Schools")
AMR_DIR = PROJECT_ROOT / "data" / "AMR"
ACFR_DIR = PROJECT_ROOT / "data" / "ACFR"
YEARS = range(2019, 2026)

amr_files = {yr: AMR_DIR / str(yr) / "middletown_amr.pdf" for yr in YEARS}
for yr, p in amr_files.items():
    print(f"{yr}: {'✓' if p.exists() else '✗'}  {p.name} ({p.stat().st_size // 1024} KB)")

2019: ✓  middletown_amr.pdf (958 KB)
2020: ✓  middletown_amr.pdf (960 KB)
2021: ✓  middletown_amr.pdf (1145 KB)
2022: ✓  middletown_amr.pdf (474 KB)
2023: ✓  middletown_amr.pdf (303 KB)
2024: ✓  middletown_amr.pdf (1119 KB)
2025: ✓  middletown_amr.pdf (410 KB)


## 1. Extract All Tables from Each AMR PDF

Use `pdfplumber` to pull every table from every page across all years.

In [2]:
def extract_tables(pdf_path: Path) -> list[tuple[int, pd.DataFrame]]:
    """Return (page_number, dataframe) for every table found in the PDF.
    
    Handles landscape/rotated pages by trying text-based table detection
    when the default strategy finds nothing.
    """
    results = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables()
            if not tables:
                tables = page.extract_tables(table_settings={
                    "vertical_strategy": "text",
                    "horizontal_strategy": "text",
                })
            for tbl in tables:
                if not tbl or len(tbl) < 2:
                    continue
                df = pd.DataFrame(tbl[1:], columns=tbl[0])
                df = df.dropna(how="all").replace("", pd.NA).dropna(how="all")
                if not df.empty:
                    results.append((page.page_number, df))
    return results

all_tables: dict[int, list[tuple[int, pd.DataFrame]]] = {}
for yr, path in amr_files.items():
    tables = extract_tables(path)
    all_tables[yr] = tables
    print(f"{yr}: {len(tables)} table(s) extracted")

print(f"\nTotal tables across all years: {sum(len(t) for t in all_tables.values())}")

2019: 14 table(s) extracted
2020: 13 table(s) extracted
2021: 14 table(s) extracted
2022: 14 table(s) extracted
2023: 11 table(s) extracted
2024: 17 table(s) extracted
2025: 17 table(s) extracted

Total tables across all years: 100


## 2. Table Extraction Summary

The AMR PDFs have complex layouts (rotated enrollment pages, split columns in narrative text).
Raw table extraction produces fragmented results. The structured data (Excess Surplus, Enrollment)
is better captured via targeted regex parsing in Sections 3 and 4 below.

In [3]:
for yr, tables in all_tables.items():
    pages = [f"p{p}" for p, _ in tables]
    print(f"{yr}: {len(tables)} table(s) on pages {', '.join(pages) if pages else '(none)'}")

2019: 14 table(s) on pages p3, p5, p7, p8, p9, p10, p11, p15, p16, p17, p19, p20, p21, p22
2020: 13 table(s) on pages p3, p5, p7, p8, p9, p10, p11, p15, p17, p19, p20, p21, p22
2021: 14 table(s) on pages p3, p5, p7, p8, p9, p10, p11, p15, p18, p19, p21, p22, p23, p24
2022: 14 table(s) on pages p3, p5, p7, p8, p9, p10, p11, p15, p18, p19, p21, p22, p23, p24
2023: 11 table(s) on pages p3, p5, p7, p8, p9, p10, p13, p17, p18, p19, p20
2024: 17 table(s) on pages p3, p5, p7, p8, p9, p10, p11, p15, p16, p17, p19, p20, p21, p23, p24, p25, p26
2025: 17 table(s) on pages p3, p5, p7, p8, p9, p10, p11, p15, p16, p17, p19, p20, p21, p23, p24, p25, p26


## 3. Excess Surplus Calculation — Multi-Year Comparison

Extract the key financial metrics from the Excess Surplus section across all years.

In [4]:
import re

# Dollar amounts in these PDFs sometimes have spaces: "$ 3 ,814,763.69"
# Use a pattern that captures digits, commas, dots, and spaces after the $
DOLLAR_RE = r"\$\s*([\d\s,]+(?:\.\d+)?)"


def parse_dollar(s: str | None) -> float | None:
    """Convert a dollar string like '3 ,814,763.69' to a float."""
    if not s:
        return None
    cleaned = re.sub(r"[^0-9.\-]", "", s)
    if not cleaned or cleaned == ".":
        return None
    try:
        return float(cleaned)
    except ValueError:
        return None


def extract_excess_surplus(pdf_path: Path) -> dict[str, float | None]:
    """Parse key values from the Excess Surplus Calculation pages."""
    with pdfplumber.open(pdf_path) as pdf:
        text = "\n".join(page.extract_text() or "" for page in pdf.pages)

    patterns = {
        "total_gen_fund_expenditures": rf"Total General Fund Expenditures per the (?:CAFR|ACFR).*?{DOLLAR_RE}",
        "on_behalf_tpaf": rf"On-Behalf (?:TPAF )?(?:Pension|TPAF).*?{DOLLAR_RE}",
        "assets_capital_leases": rf"Assets Acquired Under Capital Leases.*?{DOLLAR_RE}",
        "adjusted_expenditures": rf"Adjusted \d{{4}}-\d{{4}} General Fund Expenditures.*?{DOLLAR_RE}",
        "two_pct_threshold": rf"2% of Adjusted.*?{DOLLAR_RE}",
        "max_unassigned_fund_bal": rf"Maximum Unassigned Fund Balance.*?{DOLLAR_RE}",
        "total_fund_balance": rf"Total General Fund - Fund Balances.*?\n.*?{DOLLAR_RE}",
        "year_end_encumbrances": rf"Year-End Encumbrances.*?{DOLLAR_RE}",
        "total_unassigned_fund_bal": rf"Total Unassigned Fund Balance.*?{DOLLAR_RE}",
        "total_excess_surplus": rf"Total Excess Surplus.*?{DOLLAR_RE}",
    }

    result = {}
    for key, pattern in patterns.items():
        m = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        result[key] = parse_dollar(m.group(1)) if m else None

    # Capital & Maintenance Reserve: grab from "Detail of Other Restricted"
    # section to avoid matching "Capital Reserve to Capital Projects Fund"
    detail_match = re.search(r"Detail of Other Restricted.*", text, re.DOTALL)
    detail_text = detail_match.group(0) if detail_match else text
    for key, label in [
        ("capital_reserve", "Capital Reserve"),
        ("maintenance_reserve", "Maintenance Reserve"),
    ]:
        m = re.search(rf"{label}\s*{DOLLAR_RE}", detail_text)
        result[key] = parse_dollar(m.group(1)) if m else None

    return result


surplus_data = {}
for yr, path in amr_files.items():
    surplus_data[yr] = extract_excess_surplus(path)
    vals = surplus_data[yr]
    found = sum(1 for v in vals.values() if v is not None)
    print(f"{yr}: {found}/{len(vals)} fields parsed")

surplus_df = pd.DataFrame(surplus_data).T
surplus_df.index.name = "fiscal_year"

surplus_df_display = surplus_df.copy()
for col in surplus_df_display.columns:
    surplus_df_display[col] = surplus_df_display[col].apply(
        lambda x: f"${x:,.0f}" if pd.notna(x) and x else "—"
    )
display(surplus_df_display)

2019: 10/12 fields parsed
2020: 11/12 fields parsed
2021: 10/12 fields parsed
2022: 10/12 fields parsed
2023: 10/12 fields parsed
2024: 10/12 fields parsed
2025: 11/12 fields parsed


,total_gen_fund_expenditures,on_behalf_tpaf,assets_capital_leases,adjusted_expenditures,two_pct_threshold,max_unassigned_fund_bal,total_fund_balance,year_end_encumbrances,total_unassigned_fund_bal,total_excess_surplus,capital_reserve,maintenance_reserve
fiscal_year,,,,,,,,,,,,
2019,"$187,655,673","$26,817,338",—,"$160,838,335","$3,216,767","$3,814,764","$8,614,620","$1,228,606","$3,814,751",—,"$1,489,491","$555,533"
2020,"$185,986,835","$27,233,852",—,"$158,752,983","$3,175,060","$3,414,061","$13,519,324","$3,411,700","$4,403,360","$989,299","$2,354,491","$1,060,533"
2021,"$195,469,004","$33,501,770",—,"$161,967,234",—,"$7,655,023","$19,722,585","$2,048,734","$8,054,697","$1,388,973","$3,858,138","$2,060,533"
2022,"$213,632,833","$42,075,999",—,"$171,556,834",—,"$7,627,535","$17,853,038","$2,598,963","$6,466,258","$399,674","$2,220,917","$569,081"
2023,"$217,969,447","$42,655,573",—,"$175,313,874","$3,506,277","$3,945,082","$16,474,603","$4,239,941","$3,834,966",—,"$30,915","$569,081"
2024,"$227,451,099","$44,995,018",—,"$182,456,081","$3,649,122","$3,939,414","$7,827,613","$2,010,054","$2,903,167",—,"$240,817","$584,858"
2025,"$229,382,206","$43,367,255","$1,261,965","$184,752,986","$3,695,060","$3,955,940","$4,007,060","$376,590","$3,090,660",—,"$42,317","$36,358"


## 4. Enrollment Trends

Extract enrollment totals from the Schedule of Audited Enrollments across all years.

In [5]:
import re


def extract_enrollment(pdf_path: Path) -> dict[str, int | None]:
    """Parse enrollment totals from the Schedule of Audited Enrollments.

    The enrollment pages are often rotated 90°. After reversing each line,
    the layout reads right-to-left, so numbers appear BEFORE their labels:
        '9,421 133 9,421 Totals'
    We search for numbers preceding 'Subtotal' and 'Totals'.
    """
    with pdfplumber.open(pdf_path) as pdf:
        enrollment_text = ""
        for page in pdf.pages:
            raw = page.extract_text() or ""
            if "TCIRTSID" in raw:
                raw = "\n".join(line[::-1] for line in raw.split("\n"))
                enrollment_text += " ".join(raw.split()) + " "

    def parse_int(s: str) -> int:
        return int(s.replace(",", "").replace(" ", ""))

    # Pattern: number(s) before Subtotal/Totals (rotated text reads R-to-L)
    # "7,799 39 7,799 Subtotal" -> first number is the on-roll count
    subtotals = re.findall(r"([\d,]+)\s+\d+\s+[\d,]+\s+Subtotal", enrollment_text)
    # 3-number Totals (with shared-time): enrollment total
    totals_3 = re.findall(r"([\d,]+)\s+\d+\s+[\d,]+\s+Totals", enrollment_text)
    # 2-number Totals: low-income, transportation, LEP totals
    totals_2 = re.findall(r"([\d,]+)\s+[\d,]+\s+Totals", enrollment_text)

    # Fallback for non-rotated pages
    if not totals_3 and not subtotals:
        with pdfplumber.open(pdf_path) as pdf:
            normal_text = "\n".join(page.extract_text() or "" for page in pdf.pages)
        subtotals = re.findall(r"Subtotal\s+([\d,]+)", normal_text)
        totals_3 = re.findall(r"Totals\s+([\d,]+)", normal_text)

    regular = parse_int(subtotals[0]) if len(subtotals) >= 1 else None
    sped = parse_int(subtotals[1]) if len(subtotals) >= 2 else None
    grand = parse_int(totals_3[0]) if len(totals_3) >= 1 else None
    # Transportation total is the 2nd-largest in the 2-number totals
    transport_candidates = sorted(
        [parse_int(t) for t in totals_2 if parse_int(t) > 100],
        reverse=True,
    )
    transport = transport_candidates[0] if transport_candidates else None

    return {
        "regular_enrollment": regular,
        "special_ed_enrollment": sped,
        "total_enrollment": grand,
        "transported_students": transport,
    }


enrollment_data = {}
for yr, path in amr_files.items():
    try:
        data = extract_enrollment(path)
    except Exception as e:
        print(f"{yr}: ERROR — {e}")
        data = {k: None for k in ["regular_enrollment", "special_ed_enrollment", "total_enrollment", "transported_students"]}
    enrollment_data[yr] = data
    found = sum(1 for v in data.values() if v is not None)
    print(f"{yr}: {found}/{len(data)} fields — enrollment: {data.get('total_enrollment', '?')}, transport: {data.get('transported_students', '?')}")

enrollment_df = pd.DataFrame(enrollment_data).T
enrollment_df.index.name = "fiscal_year"
display(enrollment_df)

2019: 4/4 fields — enrollment: 9421, transport: 4247
2020: 4/4 fields — enrollment: 9379, transport: 5720
2021: 4/4 fields — enrollment: 9018, transport: 5591
2022: 4/4 fields — enrollment: 8951, transport: 5838
2023: 4/4 fields — enrollment: 8795, transport: 5388
2024: 4/4 fields — enrollment: 8680, transport: 3745
2025: 4/4 fields — enrollment: 8586, transport: 3835


,regular_enrollment,special_ed_enrollment,total_enrollment,transported_students
fiscal_year,,,,
2019,7799,1622,9421,4247
2020,7702,1677,9379,5720
2021,7377,1641,9018,5591
2022,7294,1657,8951,5838
2023,7166,1629,8795,5388
2024,7089,1591,8680,3745
2025,7056,1530,8586,3835


## 5. Visualize Key Trends

In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Total General Fund Expenditures",
        "Total Fund Balance",
        "Enrollment Trend",
        "Reserves (Capital + Maintenance)",
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.1,
)

years = list(YEARS)

fig.add_trace(
    go.Bar(
        x=years,
        y=[surplus_data[yr].get("total_gen_fund_expenditures") for yr in years],
        name="Gen Fund Expenditures",
        marker_color="#2563eb",
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Bar(
        x=years,
        y=[surplus_data[yr].get("total_fund_balance") for yr in years],
        name="Total Fund Balance",
        marker_color="#16a34a",
    ),
    row=1, col=2,
)

fig.add_trace(
    go.Scatter(
        x=years,
        y=[enrollment_data[yr].get("total_enrollment") for yr in years],
        name="Total Enrollment",
        mode="lines+markers",
        line=dict(color="#dc2626", width=3),
    ),
    row=2, col=1,
)

cap_res = [surplus_data[yr].get("capital_reserve", 0) or 0 for yr in years]
maint_res = [surplus_data[yr].get("maintenance_reserve", 0) or 0 for yr in years]
fig.add_trace(
    go.Bar(x=years, y=cap_res, name="Capital Reserve", marker_color="#7c3aed"),
    row=2, col=2,
)
fig.add_trace(
    go.Bar(x=years, y=maint_res, name="Maintenance Reserve", marker_color="#f59e0b"),
    row=2, col=2,
)

fig.update_layout(
    height=700,
    title_text="Middletown Township School District — AMR Financial Trends (FY 2019–2025)",
    showlegend=False,
    template="plotly_white",
)
fig.update_yaxes(tickformat="$,.0f", row=1, col=1)
fig.update_yaxes(tickformat="$,.0f", row=1, col=2)
fig.update_yaxes(tickformat=",.0f", row=2, col=1)
fig.update_yaxes(tickformat="$,.0f", row=2, col=2)
fig.show()

## 6. Full Text Extraction (for ad-hoc search)

Raw text from every AMR, stored in a dict for easy keyword searching.

In [7]:
amr_text: dict[int, str] = {}
for yr, path in amr_files.items():
    pages = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            raw = page.extract_text() or ""
            if "TCIRTSID" in raw:
                raw = "\n".join(line[::-1] for line in raw.split("\n"))
            pages.append(raw)
    amr_text[yr] = "\n".join(pages)
    print(f"{yr}: {len(amr_text[yr]):,} chars, {amr_text[yr].count(chr(10)):,} lines")

2019: 22,596 chars, 1,404 lines
2020: 23,012 chars, 1,439 lines
2021: 30,136 chars, 1,590 lines
2022: 30,766 chars, 1,561 lines
2023: 24,316 chars, 1,441 lines
2024: 30,432 chars, 1,612 lines
2025: 31,267 chars, 1,614 lines


In [8]:
def search_amr(keyword: str, context_lines: int = 2):
    """Search all AMR texts for a keyword, showing surrounding context."""
    for yr, text in amr_text.items():
        lines = text.split("\n")
        for i, line in enumerate(lines):
            if keyword.lower() in line.lower():
                start = max(0, i - context_lines)
                end = min(len(lines), i + context_lines + 1)
                snippet = "\n".join(f"  {lines[j]}" for j in range(start, end))
                print(f"\n[{yr}] line {i+1}:")
                print(snippet)

search_amr("finding")


[2019] line 4:
  Middletown, New Jersey
  County of Monmouth
  Auditor’s Management Report on Administrative Findings -
  Financial, Compliance and Performance
  YEAR ENDED JUNE 30, 2019

[2019] line 8:
  YEAR ENDED JUNE 30, 2019
  
  MANAGEMENT REPORT ON ADMINISTRATIVE FINDINGS
  FINANCIAL, COMPLIANCE AND PERFORMANCE
  TABLE OF CONTENTS

[2019] line 13:
  PAGE
  Report of Independent Auditors - Auditor’s Management Report on
  Administrative Findings, Financial Compliance and Performance 1
  Scope of Audit 3
  Administrative Practices and Procedures:

[2019] line 40:
  Facilities and Capital Assets 6
  Miscellaneous 6
  Follow-up on Prior Year Findings 7
  Office of Fiscal Accountability and Compliance (OFAC) Findings 7
  Acknowledgment 7

[2019] line 41:
  Miscellaneous 6
  Follow-up on Prior Year Findings 7
  Office of Fiscal Accountability and Compliance (OFAC) Findings 7
  Acknowledgment 7
  Additional Information:

[2019] line 49:
  
  AUDITOR'S MANAGEMENT REPORT ON ADMINISTRATI